# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)
df_trips.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)



# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [5]:
#Add a column that creates a unique key to identify each record in order to answer questions about individual trips
df_trips = df_trips.withColumn("trip_id", F.monotonically_increasing_id())

df_trips.select("trip_id").show(5)


+-----------+
|    trip_id|
+-----------+
|25769803776|
|25769803777|
|25769803778|
|25769803779|
|25769803780|
+-----------+
only showing top 5 rows


In [6]:
#Which trip has the highest passanger count
(
df_trips.orderBy(F.col("passenger_count").desc())
    .select("trip_id", "passenger_count", "tpep_pickup_datetime")
    .show(5)
)

+-----------+---------------+--------------------+
|    trip_id|passenger_count|tpep_pickup_datetime|
+-----------+---------------+--------------------+
|25771815874|            9.0| 2019-01-10 00:43:10|
|25777090459|            9.0| 2019-01-30 18:34:12|
|25772687771|            9.0| 2019-01-13 04:13:24|
|25771100063|            9.0| 2019-01-07 03:19:36|
|25774338483|            9.0| 2019-01-19 16:45:25|
+-----------+---------------+--------------------+
only showing top 5 rows


Strange to have a max of 9 passenger if its a normal 4 places cab 

In [7]:
#What is the Average passanger count

df_trips.select(F.avg("passenger_count").alias("avg_passenger")).show()

+------------------+
|     avg_passenger|
+------------------+
|1.5670317144945614|
+------------------+



In [8]:
#Shortest/longest trip by distance?
print("shorters trip by distance:")
(
    #add a filter to dont have trip with 0 distance who can be a mistake or canceled courses 
df_trips.filter(F.col("trip_distance") > 0)
    .orderBy("trip_distance")
    .select("trip_id", "trip_distance")
    .show(5)
)

print ("longest trip by distance:")
(
df_trips.orderBy(F.col("trip_distance").desc())
    .select("trip_id", "trip_distance")
    .show(5)
)

shorters trip by distance:
+-----------+-------------+
|    trip_id|trip_distance|
+-----------+-------------+
|25769822604|         0.01|
|25769826243|         0.01|
|25769822605|         0.01|
|25769816340|         0.01|
|25769823993|         0.01|
+-----------+-------------+
only showing top 5 rows
longest trip by distance:
+-----------+-------------+
|    trip_id|trip_distance|
+-----------+-------------+
|25775877867|        831.8|
|25774090409|        700.7|
|25776574761|       214.01|
|25774511310|       211.36|
|25774685561|       201.27|
+-----------+-------------+
only showing top 5 rows


In [9]:
#Shortest/longest trip by time?
#add a column for the trip duration in minute
df_trips = df_trips.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime")
     - F.unix_timestamp("tpep_pickup_datetime")) / 60
)
print("shortest trip by time:")
(
    #same thing than the distance
df_trips.filter(F.col("trip_duration_min") > 0) 
  .orderBy("trip_duration_min") 
  .select("trip_id", "trip_duration_min").show(5)
)
print("longest trip by time:")
(
df_trips.orderBy(F.col("trip_duration_min").desc()) 
  .select("trip_id", "trip_duration_min").show(5)
)

shortest trip by time:
+-----------+--------------------+
|    trip_id|   trip_duration_min|
+-----------+--------------------+
|25769835296|0.016666666666666666|
|25769870852|0.016666666666666666|
|25769837295|0.016666666666666666|
|25769827127|0.016666666666666666|
|25769847343|0.016666666666666666|
+-----------+--------------------+
only showing top 5 rows
longest trip by time:
+-----------+------------------+
|    trip_id| trip_duration_min|
+-----------+------------------+
|25769872043| 43648.01666666667|
|25770396038|33856.683333333334|
|25770679632|           31532.1|
|25773519501|            7190.9|
|25771519041|1687.0333333333333|
+-----------+------------------+
only showing top 5 rows


The shortest and longest trip by time and distance are outliers 

In [10]:
#busiest day/slowest single day
daily = df_trips.withColumn("trip_date", F.to_date("tpep_pickup_datetime")) \
           .groupBy("trip_date").count()

# busiest day
daily.orderBy(F.col("count").desc()).show(5) 
# slowest day
daily.orderBy(F.col("count").asc()).show(5)    

+----------+------+
| trip_date| count|
+----------+------+
|2019-01-25|292499|
|2019-01-11|291714|
|2019-01-31|284625|
|2019-01-17|284580|
|2019-01-24|281959|
+----------+------+
only showing top 5 rows
+----------+-----+
| trip_date|count|
+----------+-----+
|2019-02-23|    1|
|2019-05-20|    1|
|2019-08-13|    1|
|2019-07-23|    1|
|2018-12-21|    1|
+----------+-----+
only showing top 5 rows


In the slowest we have a mistake with a date of 2018 and some are outside 01-2019

In [11]:
#busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
hourly = (df_trips.withColumn("hour", F.hour("tpep_pickup_datetime")) 
            .groupBy("hour").count())

hourly.orderBy(F.col("count").desc()).show(5)


df_periods = df_trips.withColumn(
    "period",
    F.when((F.hour("tpep_pickup_datetime") >= 5) & (F.hour("tpep_pickup_datetime") < 12), "morning")
     .when((F.hour("tpep_pickup_datetime") >= 12) & (F.hour("tpep_pickup_datetime") < 17), "afternoon")
     .when((F.hour("tpep_pickup_datetime") >= 17) & (F.hour("tpep_pickup_datetime") < 21), "evening")
     .otherwise("late_night")
)
df_periods.groupBy("period").count().orderBy(F.col("count").desc()).show()

+----+------+
|hour| count|
+----+------+
|  18|515390|
|  19|475186|
|  17|468479|
|  15|452691|
|  14|433139|
+----+------+
only showing top 5 rows
+----------+-------+
|    period|  count|
+----------+-------+
| afternoon|2111999|
|   morning|2035497|
|   evening|1882211|
|late_night|1666910|
+----------+-------+



In [12]:
#On average which day of the week is slowest/busiest
day_of_week = (df_trips.withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE")) #EEEE is for put the name of the day and not a number  
         .groupBy("day_of_week").count() 
         .orderBy(F.col("count").desc()))
day_of_week.show(7)

+-----------+-------+
|day_of_week|  count|
+-----------+-------+
|   Thursday|1357043|
|  Wednesday|1265264|
|    Tuesday|1209084|
|     Friday|1087215|
|   Saturday|1009985|
|     Monday| 908121|
|     Sunday| 859905|
+-----------+-------+



In [13]:
#Does trip distance or num passangers affect tip amount
#stat.corr give a correlation number between -1 and 1
print("Correlation distance / tips :",
      df_trips.stat.corr("trip_distance", "tip_amount"))
print("Correlation nb passangers / tips :",
      df_trips.stat.corr("passenger_count", "tip_amount"))

Correlation distance / tips : 0.5269200663652668
Correlation nb passangers / tips : 0.004431051585116288


the correlation distance / pourboire is positive so we have a correlation so the longest the trips is the highest the tips is.
But for nb passangers / tips is near 0 so no correlation

In [14]:
#What was the highest "extra" charge and which trip
(
df_trips.orderBy(F.col("extra").desc()) 
  .select("trip_id", "extra", "fare_amount", "tpep_pickup_datetime") 
  .show(5)
)

+-----------+------+-----------+--------------------+
|    trip_id| extra|fare_amount|tpep_pickup_datetime|
+-----------+------+-----------+--------------------+
|25775127259|535.38|  355676.98| 2019-01-23 08:58:09|
|25777257006| 23.04|        4.5| 2019-01-31 10:06:09|
|25770114828|  18.5|       52.0| 2019-01-02 16:33:28|
|25772258862|  18.5|       49.0| 2019-01-11 16:08:48|
|25769938325|  18.5|       39.5| 2019-01-01 16:09:32|
+-----------+------+-----------+--------------------+
only showing top 5 rows


535 look likes a outliers 

In [15]:
#Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?
df_trips.select(F.min("trip_distance"), F.max("trip_distance"),
           F.min("trip_duration_min"), F.max("trip_duration_min"),
           F.min("fare_amount"), F.max("fare_amount")).show()

+------------------+------------------+----------------------+----------------------+----------------+----------------+
|min(trip_distance)|max(trip_distance)|min(trip_duration_min)|max(trip_duration_min)|min(fare_amount)|max(fare_amount)|
+------------------+------------------+----------------------+----------------------+----------------+----------------+
|               0.0|             831.8|              -84280.5|     43648.01666666667|          -362.0|       623259.86|
+------------------+------------------+----------------------+----------------------+----------------+----------------+



Several data quality issues were identified.

A trip distance of 831.8 miles is clearly unrealistic for New York.

A trip duration of around 30 days is probably caused by an incorrect timestamp.

Some pickup dates are outside January 2019, even though the dataset is supposed to cover this month.

An extra charge of $535.38 is an obvious outlier

A trip with 9 passengers is unusual but can still be valid.

These issues can be detected by sorting numerical columns or using df.describe() to check extreme values.

The raw data should not necessarily be deleted, but obvious outliers can be excluded from specific analyses using filters.


### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [16]:
# set download url for taxi zone lookup data
download_zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

# get the data
response = requests.get(download_zone_url)

# check that response was good and save the data
zone_file = "taxi_zone_lookup.csv"

if response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(response.content)

# read the CSV with Spark
zones = spark.read.option("header", "true").option("inferSchema", "true").csv(zone_file)

zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [17]:
# Give each dataframe an alias to avoid column name conflicts
# We use the zones table twice: once for pickup and once for dropoff
trips = df_trips.alias("t")
zones_pu = zones.alias("zpu")
zones_do = zones.alias("zdo")

# Join the trips with the zones table using the pickup location ID
df_full = (trips 
    .join(zones_pu, F.col("t.PULocationID") == F.col("zpu.LocationID")) 
    .withColumnRenamed("Borough", "pickup_borough") 
    .drop("zpu.LocationID", "Zone", "service_zone") 
    
    # Join again with the zones table using the dropoff location ID
    .join(zones_do, F.col("t.DOLocationID") == F.col("zdo.LocationID")) 
    .withColumnRenamed("Borough", "dropoff_borough") 
    .drop("zdo.LocationID", "Zone", "service_zone"))

df_full.select("trip_id", "pickup_borough", "dropoff_borough").show(5)

+-----------+--------------+---------------+
|    trip_id|pickup_borough|dropoff_borough|
+-----------+--------------+---------------+
|25769803776|     Manhattan|      Manhattan|
|25769803777|     Manhattan|      Manhattan|
|25769803778|     Manhattan|      Manhattan|
|25769803779|        Queens|         Queens|
|25769803780|        Queens|         Queens|
+-----------+--------------+---------------+
only showing top 5 rows


In [18]:
#which borough had most pickups? dropoffs?
(df_full.groupBy("pickup_borough").count() 
       .orderBy(F.col("count").desc()).show()
)

(df_full.groupBy("dropoff_borough").count() 
       .orderBy(F.col("count").desc()).show()
)

+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|6950965|
|        Queens| 471173|
|       Unknown| 159815|
|      Brooklyn|  91905|
|         Bronx|  18062|
|           N/A|   3890|
|           EWR|    446|
| Staten Island|    361|
+--------------+-------+

+---------------+-------+
|dropoff_borough|  count|
+---------------+-------+
|      Manhattan|6817355|
|         Queens| 340972|
|       Brooklyn| 301105|
|        Unknown| 149097|
|          Bronx|  58085|
|            N/A|  16904|
|            EWR|  10914|
|  Staten Island|   2185|
+---------------+-------+



In [19]:
#what are the busy/slow times by borough
(df_full.withColumn("hour", F.hour("tpep_pickup_datetime")) 
       .groupBy("pickup_borough", "hour").count() 
       .orderBy("pickup_borough", F.col("count").desc()) 
       .show(5)
)
(df_full.withColumn("hour", F.hour("tpep_pickup_datetime")) 
       .groupBy("pickup_borough", "hour").count() 
       .orderBy("pickup_borough", F.col("count").asc()) 
       .show(5)
)

+--------------+----+-----+
|pickup_borough|hour|count|
+--------------+----+-----+
|         Bronx|   7| 1803|
|         Bronx|   8| 1445|
|         Bronx|   6| 1301|
|         Bronx|   9| 1158|
|         Bronx|  10| 1079|
+--------------+----+-----+
only showing top 5 rows
+--------------+----+-----+
|pickup_borough|hour|count|
+--------------+----+-----+
|         Bronx|   2|  225|
|         Bronx|   3|  225|
|         Bronx|   1|  254|
|         Bronx|   0|  324|
|         Bronx|  22|  353|
+--------------+----+-----+
only showing top 5 rows


In [20]:
#what are the busiest days of the week by borough?
(df_full.withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE")) 
       .groupBy("pickup_borough", "day_of_week").count() 
       .orderBy("pickup_borough", F.col("count").desc()) 
       .show()
)

+--------------+-----------+-----+
|pickup_borough|day_of_week|count|
+--------------+-----------+-----+
|         Bronx|   Thursday| 3121|
|         Bronx|    Tuesday| 3059|
|         Bronx|  Wednesday| 2999|
|         Bronx|     Friday| 2666|
|         Bronx|     Monday| 2177|
|         Bronx|     Sunday| 2112|
|         Bronx|   Saturday| 1928|
|      Brooklyn|    Tuesday|15779|
|      Brooklyn|   Thursday|15714|
|      Brooklyn|  Wednesday|15101|
|      Brooklyn|     Friday|13092|
|      Brooklyn|   Saturday|11604|
|      Brooklyn|     Sunday|11099|
|      Brooklyn|     Monday| 9516|
|           EWR|  Wednesday|   83|
|           EWR|    Tuesday|   77|
|           EWR|     Friday|   74|
|           EWR|     Sunday|   68|
|           EWR|   Thursday|   58|
|           EWR|   Saturday|   55|
+--------------+-----------+-----+
only showing top 20 rows


In [21]:
#what is the average trip distance by borough?
(df_full.groupBy("pickup_borough").agg(
    F.avg("trip_distance").alias("avg_distance")
).orderBy(F.col("avg_distance").desc()).show())

+--------------+------------------+
|pickup_borough|      avg_distance|
+--------------+------------------+
| Staten Island|12.503601108033246|
|        Queens|11.283218499361993|
|         Bronx| 7.233194552098303|
|      Brooklyn| 4.787677275447492|
|           N/A| 3.193850899742941|
|           EWR| 2.641098654708519|
|       Unknown| 2.415464130400774|
|     Manhattan|2.2286693358402596|
+--------------+------------------+



In [22]:
#what is the average trip fare by borough?
(df_full.groupBy("pickup_borough").agg(
    F.avg("fare_amount").alias("avg_fare")
).orderBy(F.col("avg_fare").desc()).show())

+--------------+------------------+
|pickup_borough|          avg_fare|
+--------------+------------------+
|           EWR| 76.24024663677126|
|           N/A|  59.5731593830335|
| Staten Island|45.289861495844896|
|        Queens| 35.14462651722029|
|         Bronx| 26.26890543682963|
|      Brooklyn|18.649132800172286|
|       Unknown|14.944423051653523|
|     Manhattan|10.792468572351568|
+--------------+------------------+



In [25]:
#highest/lowest faire amounts for a trip, what burough is associated with the each
(df_full.orderBy(F.col("fare_amount").desc()) 
       .select("trip_id", "fare_amount", "pickup_borough").show(5))

(df_full.filter(F.col("fare_amount") > 0) # because 0 is a outliers 
       .orderBy("fare_amount") 
       .select("trip_id", "fare_amount", "pickup_borough").show(5))

+-----------+-----------+--------------+
|    trip_id|fare_amount|pickup_borough|
+-----------+-----------+--------------+
|25772303431|  623259.86|     Manhattan|
|25775127259|  355676.98|     Manhattan|
|25771963747|    36090.3|       Unknown|
|25771696557|   34674.65|       Unknown|
|25771453227|   33023.53|       Unknown|
+-----------+-----------+--------------+
only showing top 5 rows
+-----------+-----------+--------------+
|    trip_id|fare_amount|pickup_borough|
+-----------+-----------+--------------+
|25769814043|       0.01|     Manhattan|
|25769827670|       0.01|     Manhattan|
|25769815296|       0.01|           N/A|
|25769825313|       0.01|     Manhattan|
|25769821147|       0.01|     Manhattan|
+-----------+-----------+--------------+
only showing top 5 rows


In the lowest we can see new outiliers 

In [27]:
#load the dataset from the most recently available january, is there a change to any of the average metrics.
# set download url for January 2025 trip data
download_url_2025 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'

# get the data
response = requests.get(download_url_2025)

# check that response was good and save the data
jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"
if response.status_code == 200:
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response.content)

# Read the data with Spark
df_2025 = spark.read.parquet(jan_2025_trip_data)

# Compare 2019 and 2025
print("Nb courses 2019 vs 2025 :", df_trips.count(), "vs", df_2025.count())

print("Tarif moyen 2019 vs 2025 :",
      df_trips.select(F.avg("fare_amount")).first()[0], "vs",
      df_2025.select(F.avg("fare_amount")).first()[0])

print("Distance moy. 2019 vs 2025 :",
      df_trips.select(F.avg("trip_distance")).first()[0], "vs",
      df_2025.select(F.avg("trip_distance")).first()[0])

Nb courses 2019 vs 2025 : 7696617 vs 3475226
Tarif moyen 2019 vs 2025 : 12.52967677747685 vs 17.08180276045484
Distance moy. 2019 vs 2025 : 2.8301461681153532 vs 5.855126178843539


### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [29]:
#we create the view for using it after in the sql query
df_trips.createOrReplaceTempView("trips")
zones.createOrReplaceTempView("zones")

In [38]:
#which borough had most pickups? dropoffs?

spark.sql("""
    SELECT z.Borough AS pickup_borough, COUNT(*) AS trip_count
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY trip_count DESC
""").show()

spark.sql("""
    SELECT z.Borough AS dropoff_borough, COUNT(*) AS trip_count
    FROM trips t
    JOIN zones z ON t.DOLocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY trip_count DESC
""").show()

+--------------+----------+
|pickup_borough|trip_count|
+--------------+----------+
|     Manhattan|   6950965|
|        Queens|    471173|
|       Unknown|    159815|
|      Brooklyn|     91905|
|         Bronx|     18062|
|           N/A|      3890|
|           EWR|       446|
| Staten Island|       361|
+--------------+----------+

+---------------+----------+
|dropoff_borough|trip_count|
+---------------+----------+
|      Manhattan|   6817355|
|         Queens|    340972|
|       Brooklyn|    301105|
|        Unknown|    149097|
|          Bronx|     58085|
|            N/A|     16904|
|            EWR|     10914|
|  Staten Island|      2185|
+---------------+----------+



In [34]:
#What is the Average passanger count
spark.sql("""
    SELECT AVG(passenger_count) AS avg_passengers
    FROM trips
""").show()

+------------------+
|    avg_passengers|
+------------------+
|1.5670317144945614|
+------------------+



In [36]:
#What was the highest "extra" charge and which trip
spark.sql("""
    SELECT trip_id, extra, fare_amount
    FROM trips
    ORDER BY extra DESC
    LIMIT 5
""").show()

+-----------+------+-----------+
|    trip_id| extra|fare_amount|
+-----------+------+-----------+
|25775127259|535.38|  355676.98|
|25777257006| 23.04|        4.5|
|25772258862|  18.5|       49.0|
|25770114828|  18.5|       52.0|
|25773820911|  18.5|       91.5|
+-----------+------+-----------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing